<a href="https://colab.research.google.com/github/singhm8755/7150CEM-Ecommerce-Returns-CLV/blob/main/1_Data_Generation_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd

file_path = '/content/drive/MyDrive/7150CEM_Project/data/synthetic_ecommerce.csv'
df = pd.read_csv(file_path)


display(df.head())

,transaction_id,customer_id,transaction_date,product_category,payment_method,device_type,customer_segment,customer_tenure_days,order_frequency_12m,order_value_gbp,click_depth,time_on_page_seconds,product_page_visits,returned
0,93666,567,2024-01-01,Home_Garden,Credit_Card,Tablet,First_Time,47,1,109.19,4,271,14,0
1,68309,8327,2024-01-01,Electronics,Credit_Card,Mobile,First_Time,6,1,47.17,9,292,11,0
2,106907,7450,2024-01-01,Home_Garden,Credit_Card,Desktop,First_Time,26,1,85.01,4,121,3,0
3,87602,7738,2024-01-01,Fashion,Credit_Card,Desktop,First_Time,40,1,94.89,4,177,16,1
4,84892,11180,2024-01-01,Home_Garden,PayPal,Mobile,First_Time,49,1,91.07,10,151,14,0


In [4]:
df.to_csv('/content/drive/MyDrive/7150CEM_Project/data/synthetic_ecommerce.csv')


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random
from google.colab import drive

# Mount Google Drive to save the generated dataset
drive.mount('/content/drive')

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Dataset configuration parameters based on proposal specifications
NUM_CUSTOMERS = 12000  # Number of unique customers
NUM_TRANSACTIONS = 120000  # Total number of transactions over 24 months
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2025, 12, 31)

# Product categories with realistic return rate distributions
PRODUCT_CATEGORIES = {
    'Fashion': 0.30,      # 30% return rate for fashion items
    'Electronics': 0.15,  # 15% return rate for electronics
    'Home_Garden': 0.20   # 20% return rate for home and garden products
}

# Payment methods and their association with return behavior
PAYMENT_METHODS = ['Credit_Card', 'PayPal', 'Cash_on_Delivery']
PAYMENT_WEIGHTS = [0.60, 0.30, 0.10]  # Distribution of payment method usage

# Device types used for browsing and purchasing
DEVICE_TYPES = ['Mobile', 'Desktop', 'Tablet']
DEVICE_WEIGHTS = [0.45, 0.35, 0.20]  # Realistic device usage distribution

print("Generating customer base...")

# Generate unique customer profiles with realistic tenure distribution
customer_ids = np.arange(1, NUM_CUSTOMERS + 1)
customer_segments = np.random.choice(
    ['First_Time', 'Repeat', 'Wholesale'],
    size=NUM_CUSTOMERS,
    p=[0.40, 0.50, 0.10]  # 40% first-time, 50% repeat, 10% wholesale
)

# Customer tenure represents days since first purchase
# First-time buyers have low tenure, repeat customers have varied tenure
customer_tenure = []
for segment in customer_segments:
    if segment == 'First_Time':
        tenure = np.random.randint(1, 90)  # New customers, 1-90 days
    elif segment == 'Repeat':
        tenure = np.random.randint(90, 730)  # Repeat customers, 3 months to 2 years
    else:  # Wholesale
        tenure = np.random.randint(365, 1095)  # Established wholesale, 1-3 years
    customer_tenure.append(tenure)

# Create customer lookup table for later use
customer_df = pd.DataFrame({
    'customer_id': customer_ids,
    'segment': customer_segments,
    'tenure': customer_tenure
})

print("Generating transactions...")

# Initialize lists to store transaction-level data
transaction_data = []

for i in range(NUM_TRANSACTIONS):
    # Select a random customer for this transaction
    customer = customer_df.sample(1).iloc[0]
    customer_id = customer['customer_id']
    segment = customer['segment']
    tenure = customer['tenure']

    # Generate random transaction date within the 24-month period
    days_offset = np.random.randint(0, (END_DATE - START_DATE).days)
    transaction_date = START_DATE + timedelta(days=days_offset)

    # Select product category with realistic distribution
    category = np.random.choice(
        list(PRODUCT_CATEGORIES.keys()),
        p=[0.45, 0.35, 0.20]  # Fashion is most common category
    )
    base_return_rate = PRODUCT_CATEGORIES[category]

    # Select payment method
    payment_method = np.random.choice(PAYMENT_METHODS, p=PAYMENT_WEIGHTS)

    # Select device type
    device = np.random.choice(DEVICE_TYPES, p=DEVICE_WEIGHTS)

    # Generate behavioral features that influence return likelihood
    # Click depth: number of pages viewed before purchase
    click_depth = np.random.randint(1, 11)

    # Time on page: total seconds spent on product page
    # Higher time may indicate more informed decision, potentially lower return risk
    time_on_page = np.random.randint(5, 301)

    # Product page visits: how many times customer viewed this product
    page_visits = np.random.randint(1, 21)

    # Order value in GBP, log-normal distribution for realistic spread
    order_value = np.round(np.random.lognormal(mean=4.5, sigma=0.8), 2)
    order_value = np.clip(order_value, 10, 1000)  # Constrain to reasonable range

    # Order frequency: number of orders this customer has made in last 12 months
    # Wholesale and repeat customers have higher frequency
    if segment == 'Wholesale':
        order_frequency = np.random.randint(20, 50)
    elif segment == 'Repeat':
        order_frequency = np.random.randint(3, 20)
    else:  # First-time
        order_frequency = 1

    # Calculate return probability based on multiple factors
    # Start with base category return rate
    return_probability = base_return_rate

    # Adjust probability based on customer segment
    if segment == 'First_Time':
        return_probability *= 1.4  # First-time buyers have higher return rate
    elif segment == 'Wholesale':
        return_probability *= 0.3  # Wholesale customers rarely return

    # Cash on delivery is associated with higher return rates
    if payment_method == 'Cash_on_Delivery':
        return_probability *= 1.5

    # Mobile purchases may have slightly higher returns due to less detailed viewing
    if device == 'Mobile':
        return_probability *= 1.1

    # Lower click depth and time on page suggest rushed decision, higher return risk
    if click_depth <= 3:
        return_probability *= 1.2
    if time_on_page < 60:
        return_probability *= 1.15

    # Cap probability at realistic maximum
    return_probability = min(return_probability, 0.70)

    # Generate binary return outcome based on calculated probability
    returned = 1 if np.random.random() < return_probability else 0

    # Store transaction record
    transaction_data.append({
        'transaction_id': i + 1,
        'customer_id': customer_id,
        'transaction_date': transaction_date,
        'product_category': category,
        'payment_method': payment_method,
        'device_type': device,
        'customer_segment': segment,
        'customer_tenure_days': tenure,
        'order_frequency_12m': order_frequency,
        'order_value_gbp': order_value,
        'click_depth': click_depth,
        'time_on_page_seconds': time_on_page,
        'product_page_visits': page_visits,
        'returned': returned
    })

    # Progress indicator every 10000 transactions
    if (i + 1) % 10000 == 0:
        print(f"Generated {i + 1} / {NUM_TRANSACTIONS} transactions")

# Convert to DataFrame
df = pd.DataFrame(transaction_data)

# Sort by transaction date for temporal realism
df = df.sort_values('transaction_date').reset_index(drop=True)

print("\nDataset generation complete.")
print(f"Total transactions: {len(df)}")
print(f"Total unique customers: {df['customer_id'].nunique()}")
print(f"Overall return rate: {df['returned'].mean():.2%}")
print("\nReturn rate by category:")
print(df.groupby('product_category')['returned'].mean())

# Create output directory if it does not exist
import os
output_dir = '/content/drive/MyDrive/7150CEM_Project/data'
os.makedirs(output_dir, exist_ok=True)

# Save dataset to CSV
output_path = f'{output_dir}/synthetic_ecommerce.csv'
df.to_csv(output_path, index=False)
print(f"\nDataset saved to: {output_path}")

# Display first few rows for verification
print("\nFirst 5 rows of generated dataset:")
print(df.head())

# Display dataset summary statistics
print("\nDataset summary:")
print(df.describe())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Generating customer base...
Generating transactions...
Generated 10000 / 120000 transactions
Generated 20000 / 120000 transactions
Generated 30000 / 120000 transactions
Generated 40000 / 120000 transactions
Generated 50000 / 120000 transactions
Generated 60000 / 120000 transactions
Generated 70000 / 120000 transactions
Generated 80000 / 120000 transactions
Generated 90000 / 120000 transactions
Generated 100000 / 120000 transactions
Generated 110000 / 120000 transactions
Generated 120000 / 120000 transactions

Dataset generation complete.
Total transactions: 120000
Total unique customers: 12000
Overall return rate: 29.46%

Return rate by category:
product_category
Electronics    0.196448
Fashion        0.387232
Home_Garden    0.256005
Name: returned, dtype: float64

Dataset saved to: /content/drive/MyDrive/7150CEM_Project/data/synthetic_ecommerce.csv

First 5 